# Data Hub

This computational notebook provides an interactive framework for exploring NEON (National Ecological Observatory Network) weather station datasets alongside CLM (Community Land Model) outputs. By integrating observed meteorological data with model simulations, this environment allows for a comparative analysis of land-atmosphere interactions. The following instructions are designed to guide you through an initial assessment of the data, demonstrating the powerful analytical and visualization tools available within this workspace.

In [ ]:
# s3fs and tqdm are pre-installed in the exsoil-nsf-prototype-refactor image.
# (No pip install needed; this cell is intentionally a no-op.)


In [ ]:
1+1

In [ ]:
# cartopy, boto3, dask, netcdf4 are pre-installed in the exsoil-nsf-prototype-refactor
# image. Skipping the previously-slow pip install.
import time

start_time = time.perf_counter()
for pkg in ["cartopy", "boto3", "dask", "netCDF4"]:
    __import__(pkg)
print(f"All deps already present. Verified in {time.perf_counter() - start_time:.4f}s")


In [ ]:
import xarray as xr
print("h5netcdf" in xr.backends.list_engines())

In [ ]:
#Import Libraries
%matplotlib inline

import os
import sys
import time
import datetime

import numpy as np
import pandas as pd
import xarray as xr

from glob import glob
from os.path import join, expanduser

import matplotlib
import matplotlib.pyplot as plt

from scipy import stats

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import requests

## Weather Stations

We allow 5 stations from now: please select from the list ['ABBY','CLBJ','CPER','KONZ','TALL']

In [ ]:
url = "https://data.neonscience.org/api/v0/sites"
response = requests.get(url, timeout=20)
response.raise_for_status()

data = response.json()

target_sites = {"ABBY", "CLBJ", "CPER", "KONZ", "TALL"}

records = []

for site in data["data"]:
    code = site["siteCode"]
    if code in target_sites:
        records.append({
            "station": code,
            "name": site["siteName"],
            "lat": site["siteLatitude"],
            "lon": site["siteLongitude"],
            "state": site["stateCode"],
            "domain": site["domainCode"],
        })

neon_meta = pd.DataFrame(records).sort_values("station")
neon_meta


In [ ]:
# Station metadata (static, reproducible)
stations = neon_meta

# Create map
fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

# Geographic context
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, edgecolor="gray", linestyle=":")

# CONUS extent
ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())

# Plot stations
ax.scatter(
    stations.lon,
    stations.lat,
    s=140,
    marker="*",
    color="orange",
    edgecolor="black",
    transform=ccrs.PlateCarree(),
    zorder=5,
)

# Labels
for _, row in stations.iterrows():
    ax.text(
        row.lon + 0.6,
        row.lat + 0.6,
        row.station,
        transform=ccrs.PlateCarree(),
        fontsize=11,
        fontweight="bold",
        zorder=6,
    )

plt.title("Selected NEON Sites Used in Analysis", fontsize=15, pad=12)
plt.show()


## 2. Explore CTSM model data 

This step guides you through exploring the data from CLM simulations. There are countless ways of analyzing and processing model data. The below steps step through where to find model data and creates a plot to visualize some model data.

Each line above is a history file: model output written as NetCDF (`.nc`), a self-describing format that carries its own metadata for variables, dimensions and attributes.

The file name encodes the NEON site, the simulation type, the stream, and the date. The simulation here is **transient** — initialised by cycling over 2018-2019 tower meteorology, then run for the full length of available data.

**Two streams are written:**

- **`h0a`** — monthly averages, one file per simulated month, hundreds of variables.
- **`h1a`** — half-hourly values for selected variables, one file per simulated day, 48 timesteps each.

> Older output — including the reference copies used for validation — uses the unsuffixed **`h0`** and **`h1`** instead. CTSM 5.4 renamed them. You do not need to care which you have: `open_ctsm_hist` works it out from what is on disk.

Where the files land depends on which wrapper ran the simulation (`run_tower` archives flat; `run_neon_v2.py` adds site and experiment segments so a perturbed run and its control stay separate). Again, the reader probes for them, so a path is not something you have to supply.


In [ ]:
# ── Configuration ───────────────────────────────────────────────
# Sample sites: ABBY, CLBJ, CPER, KONZ, TALL
# KONZ is the project baseline and the site most likely to have a
# completed local run.
NEON_SITE = "KONZ"
YEAR = 2018


In [ ]:
# Reading simulation output. No credentials required: this reads the
# output your own runs produce, inside the container.
#
# open_ctsm_hist works out the details that vary between runs — which
# archive layout the wrapper used, whether the stream names are current
# (h1a/h0a) or legacy (h1/h0), and which NetCDF variant the files are
# stored in. To read the old S3 fixtures instead, set CTSM_DATA_SOURCE=s3
# and supply COS_ACCESS_KEY_ID / COS_SECRET_ACCESS_KEY.
from analytics_modules import (
    open_ctsm_hist,
    find_ctsm_hist_files,
    plot_soil_profile_timeseries,
)

try:
    sim_files = find_ctsm_hist_files(NEON_SITE, YEAR)
except FileNotFoundError:
    print(
        f"No simulation output found for {NEON_SITE} {YEAR}.\n\n"
        f"Run a simulation first:\n"
        f"    run_neon_v2 --neon-sites {NEON_SITE} --no-batch\n\n"
        f"Or point CTSM_OUTPUT_ROOT at an archive you already have:\n"
        f"    os.environ['CTSM_OUTPUT_ROOT'] = '/path/to/archive'\n\n"
        f"The paths searched are listed below.\n"
    )
    raise

print(f"{len(sim_files)} history files for {NEON_SITE} {YEAR}")
for path in sim_files[:10]:
    print("  ", path)
if len(sim_files) > 10:
    print(f"   ... and {len(sim_files) - 10} more")


In [ ]:
import time

start_time = time.perf_counter()

ds_ctsm = open_ctsm_hist(NEON_SITE, YEAR)

print("\nDims:", dict(ds_ctsm.sizes))
print("\nVars (first 25):", list(ds_ctsm.data_vars)[:25])
print(f"\nExecution time: {time.perf_counter() - start_time:.1f} seconds")


In [ ]:
ds_ctsm

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 8))

# First entry for soil moisture
ds_ctsm['H2OSOI'].plot(ax=axes[0, 0])
axes[0, 0].set_title('H2OSOI')

ds_ctsm['GPP'].plot(ax=axes[0, 1]) # GPP 
axes[0, 1].set_title('GPP')

#### 𝐿𝑎𝑡𝑒𝑛𝑡 𝐻𝑒𝑎𝑡 𝐹𝑙𝑢𝑥 = 𝑇𝑟𝑎𝑛𝑠𝑝𝑖𝑟𝑎𝑡𝑖𝑜𝑛 + 𝐶𝑎𝑛𝑜𝑝𝑦 𝐸𝑣𝑎𝑝𝑜𝑟𝑎𝑡𝑖𝑜𝑛 + 𝐺𝑟𝑜𝑢𝑛𝑑 𝐸𝑣𝑎𝑝𝑜𝑟𝑎𝑡𝑖𝑜𝑛
ds_ctsm['FCEV'].plot(ax=axes[0, 2])
axes[0, 2].set_title('FCEV')

# Second row
ds_ctsm['FCTR'].plot(ax=axes[1, 0])
axes[1, 0].set_title('FCTR')

ds_ctsm['FGEV'].plot(ax=axes[1, 1])
axes[1, 1].set_title('FGEV')

ds_ctsm['EFLX_LH_TOT'] = ds_ctsm['FCEV']+ds_ctsm['FCTR']+ds_ctsm['FGEV']
ds_ctsm['EFLX_LH_TOT'].plot(ax=axes[1, 2])
axes[1, 2].set_title('EFLX_LH_TOT')

# Leave the last subplot blank or turn it off


plt.tight_layout()
plt.show()


In [ ]:
ds = plot_soil_profile_timeseries(
    neon_site=NEON_SITE,
    var="TSOI",
    year=YEAR,
)
plt.show()
